# Age, VIP, and destination

Age is a whole number on every filled cell. The children sit well above one half. VIP does the opposite of the slogan "VIP means rescued," on a small count. Destination moves less than cryosleep.

Age, with 0 kept separate from the rest of childhood:

$$
\begin{aligned}
0 &: 144 \text{ of } 178 \\
1 \text{ to } 12 &: 420 \text{ of } 628 \\
13 \text{ to } 17 &: 409 \text{ of } 739 \\
18 \text{ to } 40 &: 2463 \text{ of } 5238 \\
41 \text{ or older} &: 852 \text{ of } 1731 \\
\text{blank} &: 90 \text{ of } 179
\end{aligned}
$$

Childhood through age 12 is the union of the first two bands:

$$
144 + 420 = 564, \qquad 178 + 628 = 806.
$$

VIP:

$$
\begin{aligned}
\text{False} &: 4198 \text{ of } 8291 \\
\text{True} &: 76 \text{ of } 199 \\
\text{blank} &: 104 \text{ of } 203
\end{aligned}
$$

Destination:

$$
\begin{aligned}
\text{TRAPPIST-1e} &: 2787 \text{ of } 5915 \\
\text{55 Cancri e} &: 1098 \text{ of } 1800 \\
\text{PSO J318.5-22} &: 401 \text{ of } 796 \\
\text{blank} &: 92 \text{ of } 182
\end{aligned}
$$

`55 Cancri e` is the high destination, at 1098 of 1800. It is a milder split than cryosleep's 2483 of 3037.


In [2]:
# Locate the contest tables beside this notebook, or under the course root.
from pathlib import Path
import csv

def manifest_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "13-Spaceship Titanic" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "sample_submission.csv").exists():
            return direct
    raise FileNotFoundError(name)

def read_manifest(name):
    with manifest_path(name).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

train = read_manifest("train.csv")
test = read_manifest("test.csv")
submission = read_manifest("sample_submission.csv")
print(len(train), len(test), len(submission))
print(train[0]["PassengerId"], train[0]["Name"])


8693 4277 4277
0001_01 Maham Ofracculy


In [3]:
# Whole numbers are stored as decimals, such as 39.0.
def whole(text):
    return int(float(text))

def luxury(row):
    columns = ("RoomService", "Spa", "VRDeck")
    if any(row[column] == "" for column in columns):
        return None
    return sum(whole(row[column]) for column in columns)

def cryo_rule(row):
    return row["CryoSleep"] == "True"

def decision_list(row):
    # Step 1. Cryosleep.
    if row["CryoSleep"] == "True":
        return True
    # Step 2. Awake, and the luxury bill is at least 500.
    bill = luxury(row)
    if row["CryoSleep"] == "False" and bill is not None and bill >= 500:
        return False
    # Step 3. Age at most 12.
    if row["Age"] != "" and whole(row["Age"]) <= 12:
        return True
    # Step 4. Europa, deck B or deck C.
    if row["Cabin"] != "":
        deck = row["Cabin"].split("/")[0]
        if row["HomePlanet"] == "Europa" and deck in ("B", "C"):
            return True
    # Step 5. Everyone else stays.
    return False

print("rules ready", len(train))


rules ready 8693


In [4]:
# Age bands, the child total, VIP, and destination.
def age_band(row):
    if row["Age"] == "":
        return "blank"
    age = whole(row["Age"])
    if age == 0:
        return "0"
    if age <= 12:
        return "1-12"
    if age <= 17:
        return "13-17"
    if age <= 40:
        return "18-40"
    return "41+"

expected_age = {
    "0": (144, 178),
    "1-12": (420, 628),
    "13-17": (409, 739),
    "18-40": (2463, 5238),
    "41+": (852, 1731),
    "blank": (90, 179),
}
seen = {name: [0, 0] for name in expected_age}
for row in train:
    seen[age_band(row)][row["Transported"] == "True"] += 1
for name, (transported, total) in expected_age.items():
    print(name, transported, total)
    assert seen[name] == [total - transported, transported]
assert (144 + 420, 178 + 628) == (564, 806)

def count_flag(column):
    found = {}
    for row in train:
        key = row[column]
        transported, total = found.get(key, (0, 0))
        found[key] = (transported + (row["Transported"] == "True"), total + 1)
    return found

vip = count_flag("VIP")
destination = count_flag("Destination")
print("VIP", vip)
print("Destination", destination)
assert vip["False"] == (4198, 8291)
assert vip["True"] == (76, 199)
assert vip[""] == (104, 203)
assert destination["TRAPPIST-1e"] == (2787, 5915)
assert destination["55 Cancri e"] == (1098, 1800)
assert destination["PSO J318.5-22"] == (401, 796)
assert destination[""] == (92, 182)


0 144 178
1-12 420 628
13-17 409 739
18-40 2463 5238
41+ 852 1731
blank 90 179
VIP {'False': (4198, 8291), 'True': (76, 199), '': (104, 203)}
Destination {'TRAPPIST-1e': (2787, 5915), 'PSO J318.5-22': (401, 796), '55 Cancri e': (1098, 1800), '': (92, 182)}


## Pitfall

VIP is `True` on 199 passengers, and 76 of them were transported. That is below one half. A model that treats VIP as a rescue flag has the direction backwards, and it has only 199 rows to learn it from.

## Summary

Age 0 is 144 of 178. Ages 0 through 12 together are 564 of 806. VIP true is 76 of 199. The strongest destination is 55 Cancri e, at 1098 of 1800.
